# Module 06: Model Evaluation & Cross-Validation Strategies

A single train/test split can be misleading: depending on the random seed, your test set might accidentally get the easiest or hardest samples.

Cross-Validation (CV) splits your training data into $K$ distinct folds, repeatedly training on $K-1$ folds and validating on the remaining fold. This provides a reliable estimate of model performance with a mean score and standard deviation.

### In this notebook, we compare 4 Core CV Splitters:
1. **Standard K-Fold (`KFold`)**: For balanced, independent, uniformly distributed tabular data.
2. **Stratified K-Fold (`StratifiedKFold`)**: Essential for classification tasks to preserve class proportions across every fold.
3. **Time Series Split (`TimeSeriesSplit`)**: Forward-chaining validation for temporal data (preventing future-to-past leakage).
4. **Group K-Fold (`GroupKFold`)**: For grouped/clustered data (e.g., multiple rows per patient or customer) where a single entity must never appear in both train and test splits.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    GroupKFold,
    cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)

# Helper function to visualize split boundaries across folds
def plot_cv_indices(cv_splitter, X, y=None, groups=None, title="Cross-Validation Split"):
    """Visualizes the train/validation splits across iterations."""
    fig, ax = plt.subplots(figsize=(9, 4))
    
    n_splits = cv_splitter.get_n_splits(X, y, groups)
    
    for i, (train_idx, val_idx) in enumerate(cv_splitter.split(X, y, groups)):
        indices = np.array([np.nan] * len(X))
        indices[train_idx] = 1   # Training set
        indices[val_idx] = 0     # Validation set
        
        ax.scatter(
            range(len(indices)), 
            [i + 0.5] * len(indices), 
            c=indices, 
            cmap=plt.cm.coolwarm, 
            marker='|', 
            lw=8,
            vmin=0, 
            vmax=1
        )

    ax.set_yticks([x + 0.5 for x in range(n_splits)])
    ax.set_yticklabels([f"Fold {x+1}" for x in range(n_splits)])
    ax.set_ylabel("CV Fold")
    ax.set_xlabel("Sample Index")
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(-0.2, n_splits)
    ax.set_xlim(-1, len(X))
    
    # Custom legend: Red = Train, Blue = Validation
    handles = [
        plt.Line2D([0], [0], color='#b40426', lw=4, label='Train Fold'),
        plt.Line2D([0], [0], color='#3b4cc0', lw=4, label='Validation Fold')
    ]
    ax.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    plt.show()

print("Helper visualization functions defined.")

Helper visualization functions defined.
